# Learning Hub Spark-Iceberg Demo

This notebook runs the full learning-hub metadata-driven demo against the demo Iceberg catalog.
It assumes the repository is available inside the container at `/home/iceberg/work/learning-hub` or `/workspace/learning-hub`.

In [ ]:
from pathlib import Path
import sys

candidate_roots = [
    Path('/home/iceberg/work/learning-hub'),
    Path('/workspace/learning-hub'),
]

root = next((path for path in candidate_roots if path.exists()), None)
if root is None:
    raise FileNotFoundError('Could not locate the learning-hub repository inside the container.')

sys.path.append(str(root / 'spark'))

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

def run_sql_file(relative_path: str):
    sql_text = (root / relative_path).read_text()
    statements = [part.strip() for part in sql_text.split(';') if part.strip()]
    for statement in statements:
        spark.sql(statement)

print(f'Using repo root: {root}')

In [ ]:
run_sql_file('ddl/demo_reset.sql')
run_sql_file('ddl/demo_metadata_framework.sql')
run_sql_file('ddl/demo_staging_sources.sql')
run_sql_file('ddl/demo_seed_batch_1001.sql')

spark.sql('SELECT source_name, load_mode FROM demo.ctl.source_registration ORDER BY source_name').show(truncate=False)

In [ ]:
from runtime_framework import run_pipeline_for_source

for source_name, batch_id in [
    ('customer', 1001),
    ('orders', 1101),
    ('product', 1201),
    ('invoice', 1301),
    ('shipment', 1401),
]:
    run_pipeline_for_source(spark, source_name=source_name, catalog='demo', forced_batch_id=batch_id)

spark.sql('SELECT batch_id, source_name, run_status FROM demo.ctl.pipeline_run ORDER BY batch_id').show(truncate=False)

In [ ]:
run_sql_file('ddl/demo_seed_batch_1002.sql')

for source_name, batch_id in [
    ('customer', 1002),
    ('orders', 1102),
    ('product', 1202),
    ('invoice', 1302),
    ('shipment', 1402),
]:
    run_pipeline_for_source(spark, source_name=source_name, catalog='demo', forced_batch_id=batch_id)

spark.sql('SELECT batch_id, source_name, run_status, watermark_value FROM demo.ctl.pipeline_run ORDER BY batch_id').show(truncate=False)

In [ ]:
run_sql_file('ddl/demo_sales_serving.sql')
spark.sql('SELECT * FROM demo.gold.sales_order_fulfillment_iceberg ORDER BY order_id').show(truncate=False)

In [ ]:
spark.sql('SHOW TABLES IN demo.bronze').show(truncate=False)
spark.sql('SHOW TABLES IN demo.silver').show(truncate=False)
spark.sql('SHOW TABLES IN demo.gold').show(truncate=False)

bronze_tables = [row.tableName for row in spark.sql('SHOW TABLES IN demo.bronze').collect()]
for table_name in bronze_tables:
    print(f'\n=== demo.bronze.{table_name} ===')
    spark.sql(f'SELECT * FROM demo.bronze.{table_name} LIMIT 20').show(truncate=False)

silver_tables = [row.tableName for row in spark.sql('SHOW TABLES IN demo.silver').collect()]
for table_name in silver_tables:
    print(f'\n=== demo.silver.{table_name} ===')
    spark.sql(f'SELECT * FROM demo.silver.{table_name} LIMIT 20').show(truncate=False)

In [ ]:
spark.sql("""
INSERT INTO demo.ctl.replay_request VALUES (
    9001,
    'customer',
    'FULL_REBUILD',
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    'platform-support',
    'REQUESTED',
    current_timestamp(),
    NULL,
    NULL,
    'Rebuild customer history and current from bronze'
)
""")

from runtime_framework import run_replay_request
run_replay_request(spark, replay_request_id=9001, catalog='demo')

spark.sql('SELECT replay_request_id, source_name, request_status FROM demo.ctl.replay_request ORDER BY replay_request_id').show(truncate=False)